# News Credibility Analyser
## Phase 2 — Text Preprocessing & Feature Engineering

**Input:** `cleaned.csv` from Phase 1  
**Goal:** Transform raw article text into a clean TF-IDF feature matrix ready for model training.  
**Notebook:** `02_feature_model.ipynb`

### What happens in this phase
1. Load and inspect the cleaned dataset
2. Build a `clean_text()` preprocessing pipeline
3. Apply preprocessing to produce `text_processed`
4. Split into train and test sets (80/20 stratified)
5. Fit a TF-IDF vectorizer on training data only
6. Save `train.csv`, `test.csv`, and `vectorizer.pkl`

## 1. Setup — Imports & Configuration

In [23]:
# Standard library
import re
import os
import time
import warnings
warnings.filterwarnings("ignore")

# Data manipulation
import numpy as np
import pandas as pd

# NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus   import stopwords
from nltk.stem     import WordNetLemmatizer

In [24]:
# Feature engineering
from sklearn.model_selection  import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Model persistence
import joblib

# Download required NLTK assets
nltk.download("punkt",        quiet=True)
nltk.download("punkt_tab",    quiet=True)
nltk.download("stopwords",    quiet=True)
nltk.download("wordnet",      quiet=True)
nltk.download("omw-1.4",      quiet=True)

print("All imports successful.")

All imports successful.


## 2. Mount Google Drive

In [25]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Load Cleaned Dataset

In [26]:
INPUT_PATH = "/content/drive/MyDrive/News_Credibility/data/processed/cleaned.csv"

df = pd.read_csv(INPUT_PATH)

print(f"Shape            : {df.shape}")
print(f"Columns          : {df.columns.tolist()}")
print(f"Label distribution:")
print(df["label"].value_counts().rename({0: "Credible", 1: "Fake"}).to_string())
print()
df.head(3)

Shape            : (38644, 3)
Columns          : ['title', 'text', 'label']
Label distribution:
label
Credible    21191
Fake        17453



,title,text,label
0,"BREAKING: GOP Chairman Grassley Has Had Enough, DEMANDS Trump Jr. Testimony","Donald Trump s White House is in chaos, and they are trying to cover it up. Their Russia problems are mounting by th...",1
1,Failed GOP Candidates Remembered In Hilarious Mocking Eulogies (VIDEO),"Now that Donald Trump is the presumptive GOP nominee, it s time to remember all those other candidates who tried so ...",1
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY Trolling Him For Being A Homophobic Bigot,"Mike Pence is a huge homophobe. He supports ex-gay conversion therapy, opposes hate crimes protections for LGBTQ peo...",1


## 4. Combine Title and Text

The article title often carries signals (sensational wording, ALL-CAPS, clickbait phrasing) that the body alone may not fully represent. Concatenating them with a separator token ` | ` lets the model learn from both fields in a single feature space without treating them as separate inputs.

In [27]:
df["full_text"] = df["title"] + " | " + df["text"]

print(f"Shape after adding full_text: {df.shape}")
print()
print("Sample full_text entry:")
print(df["full_text"].iloc[0][:300], "…")

Shape after adding full_text: (38644, 4)

Sample full_text entry:
 BREAKING: GOP Chairman Grassley Has Had Enough, DEMANDS Trump Jr. Testimony | Donald Trump s White House is in chaos, and they are trying to cover it up. Their Russia problems are mounting by the hour, and they refuse to acknowledge that there are problems surrounding all of this. To them, it s  fa …


## 5. Text Preprocessing

### Why preprocessing is necessary

Raw text contains noise that does not carry meaning: URLs, HTML tags, punctuation, and stop words like *the*, *is*, *at*. If left in, they consume feature budget, add variance, and dilute the signal the model needs to learn the difference between Fake and Credible writing.

Preprocessing standardises vocabulary so that *Running*, *runs*, and *ran* all map to the same root token *run*, reducing the feature space without losing information.

### Why negations are preserved

Standard NLTK stopwords include *not*, *no*, *never*, *without*, *neither*, and *nor*. Removing them would collapse *"the claim is not verified"* and *"the claim is verified"* into the same token sequence — semantically opposite sentences would become identical features. For a credibility classifier, negation is a meaningful signal and must be retained.

### Why lemmatization over stemming

Stemming is faster but produces non-words (*"running"* → *"run"*, *"studies"* → *"studi"*). Lemmatization uses vocabulary and grammar rules to return a proper dictionary form, which keeps the output human-readable and improves TF-IDF quality.

In [28]:
# ── Preprocessing constants ────────────────────────────────────────────────────

# Words to keep that NLTK would otherwise remove
NEGATIONS = {"not", "no", "never", "without", "neither", "nor"}

# Build custom stopword list: standard English list minus the negations above
STOP_WORDS = set(stopwords.words("english")) - NEGATIONS

# Single shared lemmatizer instance (creating one per call is expensive)
lemmatizer = WordNetLemmatizer()


# ── clean_text pipeline ────────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    """
    Full preprocessing pipeline applied to a single string.

    Steps (in order):
        1. Lowercase
        2. Strip Reuters wire-service dateline / source-name leak
        3. Remove URLs
        4. Remove HTML tags
        5. Replace digit sequences with NUM token
        6. Remove punctuation
        7. Collapse multiple spaces
        8. Tokenize
        9. Remove stopwords (keeping negations)
        10. Lemmatize
        11. Rejoin into a single string
    """

    # 1. Lowercase
    text = text.lower()

    # 2. Strip Reuters wire-service dateline leak.
    #    Nearly every True.csv article opens with a dateline like
    #    "washington (reuters) -" / "london (reuters) -" and Fake.csv
    #    articles almost never contain this. Left in, the literal word
    #    "reuters" becomes a near-perfect proxy for the label instead of
    #    the model learning genuine credibility signals. Strip the dateline
    #    phrase, then remove any remaining standalone "reuters" mentions.
    text = re.sub(r"^.*?\(reuters\)\s*-\s*", "", text)
    text = re.sub(r"\breuters\b", "", text)

    # 3. Remove URLs (http/https and bare www addresses)
    text = re.sub(r"http\S+|www\S+", "", text)

    # 4. Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # 5. Replace digit sequences with a placeholder token
    #    Preserves the information that a number appeared without memorising values
    text = re.sub(r"\d+", "NUM", text)

    # 6. Remove punctuation — keep only word characters and whitespace
    text = re.sub(r"[^\w\s]", "", text)

    # 7. Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    # 8. Tokenize
    tokens = word_tokenize(text)

    # 9. Remove stopwords; preserve negation words defined in NEGATIONS
    tokens = [t for t in tokens if t not in STOP_WORDS]

    # 10. Lemmatize each token
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    # 11. Rejoin
    return " ".join(tokens)


print("clean_text() defined.")
print()

# Quick smoke test — confirms the Reuters leak is actually gone
sample_input  = "WASHINGTON (Reuters) - Breaking NEWS!!! Scientists find MIRACLE cure! Visit http://fake-news.com <b>Click here</b> for the 100% truth."
sample_output = clean_text(sample_input)
print(f"Input  : {sample_input}")
print(f"Output : {sample_output}")
assert "reuters" not in sample_output, "Reuters leak still present!"
print("\nReuters leak check passed — 'reuters' does not appear in cleaned output.")

clean_text() defined.

Input  : WASHINGTON (Reuters) - Breaking NEWS!!! Scientists find MIRACLE cure! Visit http://fake-news.com <b>Click here</b> for the 100% truth.
Output : breaking news scientist find miracle cure visit click NUM truth

Reuters leak check passed — 'reuters' does not appear in cleaned output.


## 6. Apply Preprocessing

Applying `clean_text()` to ~44K articles typically takes **2–4 minutes** on Colab's free tier. The progress timer below will confirm when it is done.

In [29]:
print("Applying clean_text() to all articles …")

start = time.time()
df["text_processed"] = df["full_text"].apply(clean_text)
elapsed = time.time() - start

print(f"Done — {len(df):,} articles processed in {elapsed:.1f}s")
print(f"Shape : {df.shape}")

Applying clean_text() to all articles …
Done — 38,644 articles processed in 149.5s
Shape : (38644, 5)


## 7. Before / After Comparison Table

Inspect 5 random samples to confirm the pipeline is working correctly — no URLs, no HTML, no punctuation, negations retained, numbers replaced with `NUM`.

In [30]:
# Sample 5 rows reproducibly
sample_df = df[["full_text", "text_processed"]].sample(5, random_state=42).reset_index(drop=True)

# Truncate display so the table fits the screen
pd.set_option("display.max_colwidth", 120)

sample_df.style.set_properties(**{
    "text-align": "left",
    "white-space": "pre-wrap",
    "font-size": "12px",
}).set_table_styles([
    {"selector": "th", "props": [("text-align", "left"), ("font-weight", "bold")]}
])

## 8. Train / Test Split

### Design decisions

| Decision | Value | Reason |
|---|---|---|
| Split ratio | 80 / 20 | Industry standard; leaves enough data to train on |
| `stratify=y` | Yes | Preserves the class ratio in both sets |
| `random_state` | 42 | Reproducibility |

The split is done **before** TF-IDF fitting. The vectorizer will see only training text, which means the test set is a genuinely unseen holdout — the same way the model would encounter new articles in production.

In [31]:
X = df["text_processed"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,
    stratify     = y,
    random_state = 42,
)

print(f"Training set   : {len(X_train):>6,} articles")
print(f"Test set       : {len(X_test):>6,} articles")
print()
print("Training label distribution:")
print(y_train.value_counts().rename({0: "Credible", 1: "Fake"}).to_string())
print()
print("Test label distribution:")
print(y_test.value_counts().rename({0: "Credible", 1: "Fake"}).to_string())

Training set   : 30,915 articles
Test set       :  7,729 articles

Training label distribution:
label
Credible    16953
Fake        13962

Test label distribution:
label
Credible    4238
Fake        3491


## 9. Export Train and Test CSVs

Save the split datasets so Phase 3 (model training) can load them directly without re-running preprocessing.

In [32]:
PROCESSED_DIR = "/content/drive/MyDrive/News_Credibility/data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Reconstruct DataFrames with all needed columns
train_df = df.loc[X_train.index, ["title", "text", "label", "text_processed"]].copy()
test_df  = df.loc[X_test.index,  ["title", "text", "label", "text_processed"]].copy()

train_df.to_csv(f"{PROCESSED_DIR}/train.csv", index=False)
test_df.to_csv( f"{PROCESSED_DIR}/test.csv",  index=False)

print(f"train.csv saved — shape: {train_df.shape}")
print(f"test.csv  saved — shape: {test_df.shape}")
print(f"   Directory: {PROCESSED_DIR}")

train.csv saved — shape: (30915, 4)
test.csv  saved — shape: (7729, 4)
   Directory: /content/drive/MyDrive/News_Credibility/data/processed


## 10. TF-IDF Feature Engineering

### Why TF-IDF

**Term Frequency–Inverse Document Frequency** scores each word by how often it appears in a given article (TF) discounted by how common it is across all articles (IDF). Words that appear everywhere (*said*, *the*) get low scores. Words that are distinctive to a specific article get high scores.

For fake-news detection, TF-IDF is the standard and most effective baseline because:
- Fake and Credible articles have **different vocabulary distributions** — the EDA confirmed this
- It requires no pre-trained embeddings and is fully interpretable via feature coefficients
- It trains in seconds and generalises well on datasets of this size

### Parameter choices

| Parameter | Value | Reason |
|---|---|---|
| `max_features` | 30,000 | Covers the informative vocabulary without memorising rare noise |
| `ngram_range` | (1, 2) | Captures single words and two-word phrases; phrases like *"fake news"* or *"breaking exclusive"* are strong signals |
| `min_df` | 3 | Drops terms that appear in fewer than 3 articles — likely typos or proper nouns |
| `sublinear_tf` | True | Applies `1 + log(tf)` instead of raw `tf`, preventing very frequent terms from dominating |

### Why fit only on training data

If the vectorizer is fit on the full dataset (train + test combined), the IDF values are computed using test articles. The model would then have indirect knowledge of the test distribution before evaluation — a form of **data leakage**. Fitting on training data only means the test set is a genuine holdout, and evaluation metrics reflect real-world performance.

In [33]:
# Initialise vectorizer with the exact parameters from the roadmap
vectorizer = TfidfVectorizer(
    max_features = 30000,
    ngram_range  = (1, 2),
    min_df       = 3,
    sublinear_tf = True,
)

# Fit on TRAINING data only — never on test data
print("Fitting TF-IDF vectorizer on training data …")
start     = time.time()
X_train_tfidf = vectorizer.fit_transform(X_train)
elapsed   = time.time() - start
print(f"Fit complete in {elapsed:.1f}s")

# Transform test data using the already-fitted vectorizer
print("Transforming test data …")
X_test_tfidf = vectorizer.transform(X_test)
print("Transform complete.")

Fitting TF-IDF vectorizer on training data …
Fit complete in 50.8s
Transforming test data …
Transform complete.


## 11. Verify Feature Matrix Dimensions

In [34]:
print("Feature matrix dimensions")
print("─" * 35)
print(f"X_train : {X_train_tfidf.shape}  → ({X_train_tfidf.shape[0]:,} articles × {X_train_tfidf.shape[1]:,} features)")
print(f"X_test  : {X_test_tfidf.shape}   → ({X_test_tfidf.shape[0]:,} articles × {X_test_tfidf.shape[1]:,} features)")
print()

# Sanity checks
assert X_train_tfidf.shape[1] == X_test_tfidf.shape[1], \
    "Feature count mismatch between train and test — something is wrong."
assert X_train_tfidf.shape[0] == len(y_train), \
    "Row count mismatch between X_train and y_train."
assert X_test_tfidf.shape[0] == len(y_test), \
    "Row count mismatch between X_test and y_test."

print("All dimension checks passed.")

Feature matrix dimensions
───────────────────────────────────
X_train : (30915, 30000)  → (30,915 articles × 30,000 features)
X_test  : (7729, 30000)   → (7,729 articles × 30,000 features)

All dimension checks passed.


## 12. Inspect Vocabulary — Sample Features

In [35]:
vocab = vectorizer.get_feature_names_out()

print(f"Total vocabulary size : {len(vocab):,}")
print()
print("First 20 features (unigrams, alphabetical):")
print(list(vocab[:20]))
print()
print("Sample bigram features:")
bigrams = [f for f in vocab if " " in f]
print(bigrams[:20])

Total vocabulary size : 30,000

First 20 features (unigrams, alphabetical):
['aaplo', 'aaron', 'aba', 'abadi', 'abadi said', 'abandon', 'abandoned', 'abandoning', 'abbas', 'abbasi', 'abbott', 'abc', 'abc news', 'abc week', 'abdel', 'abdel fattah', 'abdrabbu', 'abdrabbu mansour', 'abducted', 'abduction']

Sample bigram features:
['abadi said', 'abc news', 'abc week', 'abdel fattah', 'abdrabbu mansour', 'abdullah saleh', 'abe said', 'able find', 'able get', 'able make', 'able reach', 'able see', 'able take', 'able use', 'able vote', 'aboard air', 'abortion clinic', 'abortion law', 'abortion provider', 'abortion right']


## 13. Save Vectorizer

In [36]:
MODELS_DIR = "/content/drive/MyDrive/News_Credibility/models"
os.makedirs(MODELS_DIR, exist_ok=True)

VECTORIZER_PATH = f"{MODELS_DIR}/vectorizer.pkl"
joblib.dump(vectorizer, VECTORIZER_PATH)

print(f"vectorizer.pkl saved")
print(f"   Path : {VECTORIZER_PATH}")
print()

# Verify the saved file can be reloaded correctly
vectorizer_reloaded = joblib.load(VECTORIZER_PATH)
test_sentence       = "government officials said the report was not verified"
test_vec            = vectorizer_reloaded.transform([clean_text(test_sentence)])

print("Reload verification:")
print(f"  Input sentence : '{test_sentence}'")
print(f"  Vector shape   : {test_vec.shape}")
print(f"  Non-zero terms : {test_vec.nnz}")
print("Vectorizer reloads and transforms correctly.")

vectorizer.pkl saved
   Path : /content/drive/MyDrive/News_Credibility/models/vectorizer.pkl

Reload verification:
  Input sentence : 'government officials said the report was not verified'
  Vector shape   : (1, 30000)
  Non-zero terms : 10
Vectorizer reloads and transforms correctly.
